# Colab Run 1TOKENIZER=morph_bpe_8k | SEED=42 | RUN_ID=exp01_morph_bpe_8k_s42

In [ ]:
from google.colab import driveimport os, shutil, subprocessdrive.mount('/content/drive')work_root='/content'repo_dir=os.path.join(work_root,'arastudy')if os.path.exists(repo_dir): shutil.rmtree(repo_dir)os.chdir(work_root)subprocess.run(['git','clone','https://github.com/faresrafat3/arastudy'], check=True)os.chdir(repo_dir)subprocess.run(['pip','install','-r','requirements.txt','-q'], check=True)subprocess.run(['nvidia-smi'], check=True)

In [ ]:
import os, shutil, zipfilefrom pathlib import Pathrepo=Path('/content/arastudy')drive_root=Path('/content/drive/MyDrive/arastudy_data')zip_path=Path('/content/drive/MyDrive/arastudy_cloud_data.zip')if not (drive_root/'splits/phase2b').exists() or not (drive_root/'tokenizers/phase2b').exists():    if zip_path.exists():        drive_root.mkdir(parents=True, exist_ok=True)        with zipfile.ZipFile(zip_path,'r') as zf: zf.extractall('/content/drive/MyDrive')        extracted=Path('/content/drive/MyDrive/cloud_data')        if extracted.exists():            if (drive_root/'splits').exists(): shutil.rmtree(drive_root/'splits')            if (drive_root/'tokenizers').exists(): shutil.rmtree(drive_root/'tokenizers')            shutil.move(str(extracted/'splits'), str(drive_root/'splits'))            shutil.move(str(extracted/'tokenizers'), str(drive_root/'tokenizers'))            shutil.rmtree(extracted)if not (drive_root/'splits/phase2b').exists() or not (drive_root/'tokenizers/phase2b').exists():    raise RuntimeError('Dataset not ready on Drive')s=repo/'data/splits/phase2b'; t=repo/'results/tokenizers/phase2b's.parent.mkdir(parents=True,exist_ok=True); t.parent.mkdir(parents=True,exist_ok=True)if s.exists() or s.is_symlink():    try: s.unlink()    except Exception: shutil.rmtree(s)if t.exists() or t.is_symlink():    try: t.unlink()    except Exception: shutil.rmtree(t)os.symlink(str(drive_root/'splits/phase2b'), str(s))os.symlink(str(drive_root/'tokenizers/phase2b'), str(t))assert (t/'morph_bpe_8k.model').exists() and (t/'morph_bpe_8k.vocab').exists()print('✅ Dataset ready', drive_root)

In [ ]:
from pathlib import PathTOKENIZER='morph_bpe_8k'; SEED=42; RUN_ID='exp01_morph_bpe_8k_s42'; RESUME=Falserun_dir=Path(f'/content/arastudy/results/exp01/{RUN_ID}')if run_dir.exists() and not RESUME: shutil.rmtree(run_dir)cmd=['python','-m','src.training.train','--config','configs/experiments/exp01_tokenization.yaml','--tokenizer-id',TOKENIZER,'--seed',str(SEED),'--run-id',RUN_ID,'--output-dir','results/exp01','--hardware','colab']if RESUME: cmd.append('--resume')subprocess.run(cmd, check=True)

In [ ]:
import jsonfrom pathlib import Pathsrc=Path('/content/arastudy/results/exp01/exp01_morph_bpe_8k_s42')dst=Path('/content/drive/MyDrive/arastudy_results/exp01/exp01_morph_bpe_8k_s42')dst.parent.mkdir(parents=True, exist_ok=True)shutil.copytree(src, dst, dirs_exist_ok=True)summary=json.load(open(src/'summary.json', encoding='utf-8'))print(f'✅ Done! Best BPC: {summary["best_bpc"]:.4f}')